# Thermostat with Hysteresis - Hybrid Automaton Example

This notebook demonstrates a hybrid automaton modeling a traffic light controller.

**States**: HEATING → IDL → COOLING

**Transitions** occur based on time thresholds:
- RED stays for 5 seconds
- GREEN stays for 8 seconds
- YELLOW stays for 3 seconds

## 1. Import Required Modules

In [30]:
from hybrid_automaton import Automaton, State, Transition
import time
import numpy as np

## 2. Define Guard Functions

Guards check if enough time has elapsed to trigger a transition.

In [31]:
def too_cold(x, aux_x, u, ctx, dt):
    return x < 18.0

def too_hot(x, aux_x, u, ctx, dt):
    return x > 22.0

def temp_comfortable(x, aux_x, u, ctx, dt):
    return 18.5 <= x <= 21.5

## 3. Define Dynamics

The dynamic function for each state.

In [32]:
def heating_dynamics(x, aux_x, u, ctx, dt):
    """Temperature increases when heating"""
    return 2.0  # Heating rate: 2°C/s

def cooling_dynamics(x, aux_x, u, ctx, dt):
    """Temperature decreases when cooling"""
    return -1.5  # Cooling rate: -1.5°C/s

def idle_dynamics(x, aux_x, u, ctx, dt):
    """Temperature drifts toward ambient (20°C)"""
    ambient = 20.0
    drift_rate = 0.5
    return drift_rate * (ambient - x)

## 4. Define Callback Functions

callback functions for on entry of each of the states

In [33]:
def on_enter_heating():
    print("🔥 HEATING - Turning heater ON")

def on_enter_cooling():
    print("❄️  COOLING - Turning AC ON")

def on_enter_idle():
    print("😌 IDLE - Systems off, maintaining temperature")

## 5. Define States

In [34]:
heating = State(name="HEATING", flow=heating_dynamics, on_enter=on_enter_heating)
cooling = State(name="COOLING", flow=cooling_dynamics, on_enter=on_enter_cooling)
idle = State(name="IDLE", initial=True, flow=idle_dynamics, on_enter=on_enter_idle)

## 6. Define Transitions

In [35]:
idle.add_transition(Transition("idle_to_heating", heating, guards=[too_cold], priority=1))
idle.add_transition(Transition("idle_to_cooling", cooling, guards=[too_hot], priority=1))

heating.add_transition(Transition("heating_to_idle", idle, guards=[temp_comfortable], priority=1))
cooling.add_transition(Transition("cooling_to_idle", idle, guards=[temp_comfortable], priority=1))

## 7. Create Automaton

In [36]:
thermostat = Automaton(
    name="Thermostat",
    states=[heating, cooling, idle],
    dt=0.1,
    real_time_mode=False
)

# Initial state: temperature at 15°C (cold)
x0 = 15.0
thermostat.activate(x0=x0)

## 8. Run the Simulation 

In [37]:
for i in range(200):
    result = thermostat.step()
    if i % 20 == 0:
        print(f"t={i*0.1:.1f}s | State: {thermostat.q.name:8s} | Temp: {thermostat.x:.2f}°C")
    if result and result.transition_taken:
        print(f"  → Transition: {result.transition_taken.name}")

print(f"\nFinal temperature: {thermostat.x:.2f}°C")
print()

🔥 HEATING - Turning heater ON
t=0.0s | State: HEATING  | Temp: 15.25°C
  → Transition: idle_to_heating
😌 IDLE - Systems off, maintaining temperature
  → Transition: heating_to_idle
t=2.0s | State: IDLE     | Temp: 18.84°C
t=4.0s | State: IDLE     | Temp: 19.59°C
t=6.0s | State: IDLE     | Temp: 19.85°C
t=8.0s | State: IDLE     | Temp: 19.95°C
t=10.0s | State: IDLE     | Temp: 19.98°C
t=12.0s | State: IDLE     | Temp: 19.99°C
t=14.0s | State: IDLE     | Temp: 20.00°C
t=16.0s | State: IDLE     | Temp: 20.00°C
t=18.0s | State: IDLE     | Temp: 20.00°C

Final temperature: 20.00°C

